In [1]:
import os
import sys
import gc
import time
import random
import argparse
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms

from pathlib import Path
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.models import resnet18

In [2]:
# config
BASE = Path(__file__).parent
# BASE = Path.cwd()
PUB_PATH = BASE / "pub.pt"
PRIV_PATH = BASE / "priv.pt"
MODEL_PATH = BASE / "model.pt"
OUTPUT_CSV = BASE / "submission.csv"

BASE_URL  = "http://34.63.153.158"   # DO NOT CHANGE
API_KEY   = "YOUR_API"
TASK_ID   = "01-mia"                  # DO NOT CHANGE

In [3]:
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
MEAN = [0.7406, 0.5331, 0.7059]
STD  = [0.1491, 0.1864, 0.1301]

transform_eval = transforms.Compose([
    transforms.Resize(32),
    transforms.Normalize(mean=MEAN, std=STD),
])

transform_train = transforms.Compose([
    transforms.Resize(32),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.Normalize(mean=MEAN, std=STD),
])

In [5]:
# NUM_SHADOW    = 1
# SHADOW_EPOCHS = 1
# SHADOW_LR     = 0.1
# SHADOW_WD     = 1e-3
# SHADOW_LS     = 0.05
# SHADOW_BATCH  = 256
# SUBSET_RATIO  = 0.5

Shadow config: n=1, lr=0.1, ep=1, wd=0.001, ls=0.05


In [6]:
NUM_SHADOW    = 10
SHADOW_EPOCHS = 80
SHADOW_LR     = 0.1
SHADOW_WD     = 1e-3
SHADOW_LS     = 0.05
SHADOW_BATCH  = 256
SUBSET_RATIO  = 0.5

In [7]:
class TaskDataset(Dataset):
    def __init__(self, transform=None):
        self.ids = []
        self.imgs = []
        self.labels = []
        self.transform = transform

    def __getitem__(self, index):
        id_  = self.ids[index]
        img  = self.imgs[index]
        if self.transform is not None:
            img = self.transform(img)
        return id_, img, self.labels[index]

    def __len__(self):
        return len(self.ids)


class MembershipDataset(TaskDataset):
    def __init__(self, transform=None):
        super().__init__(transform)
        self.membership = []

    def __getitem__(self, index):
        id_, img, label = super().__getitem__(index)
        return id_, img, label, self.membership[index]


def custom_collate_fn(batch):

    ids  = [b[0] for b in batch]
    imgs = torch.stack([b[1] for b in batch])
    labs = torch.tensor([b[2] for b in batch])
    mems = [b[3] for b in batch]
    return ids, imgs, labs, mems


In [8]:
print("Loading datasets...")
pub_ds  = torch.load(PUB_PATH,  weights_only=False)
priv_ds = torch.load(PRIV_PATH, weights_only=False)

pub_labels   = np.array(pub_ds.labels  if isinstance(pub_ds.labels, list) else pub_ds.labels.tolist())
priv_labels  = np.array(priv_ds.labels if isinstance(priv_ds.labels, list) else priv_ds.labels.tolist())

pub_ds.transform = transform_eval
priv_ds.transform = transform_eval

Loading datasets...


In [9]:
def make_model():
    m = resnet18(weights=None)
    m.conv1  = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    m.maxpool = nn.Identity()
    m.fc     = nn.Linear(512, 9)
    return m


def get_confidences(model, dataset):
    dataset.transform = transform_eval
    loader = DataLoader(dataset, batch_size=256, shuffle=False, collate_fn=custom_collate_fn)
    confs = []
    model.eval()
    with torch.no_grad():
        for _, imgs, labels, _ in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            probs = torch.softmax(model(imgs), dim=1)
            conf  = probs[range(len(labels)), labels]
            confs.append(conf.cpu().numpy())
    return np.concatenate(confs)

In [10]:
print("Loading target model...")
target_model = make_model()
target_model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
target_model = target_model.to(DEVICE)
target_model.eval()

target_conf_pub  = get_confidences(target_model, pub_ds)
target_conf_priv = get_confidences(target_model, priv_ds)


Loading target model...


In [11]:
pub_loader  = DataLoader(pub_ds,  batch_size=64, shuffle=False, collate_fn=custom_collate_fn)
priv_loader = DataLoader(priv_ds, batch_size=64, shuffle=False, collate_fn=custom_collate_fn)

def collect_preds(loader, model):
    ids_out, preds_out, labels_out = [], [], []
    model.eval()
    with torch.no_grad():
        for ids, imgs, labels, _ in loader:
            imgs = imgs.to(DEVICE)
            predicted = torch.softmax(model(imgs), dim=1).argmax(dim=1)
            ids_out.extend(ids if isinstance(ids, list) else ids.tolist())
            preds_out.extend(predicted.cpu().numpy().tolist())
            labels_out.extend(labels.numpy().tolist())
    return pd.DataFrame({"id": ids_out, "predicted_label": preds_out, "true_label": labels_out})

results_df      = collect_preds(pub_loader,  target_model)
results_df_priv = collect_preds(priv_loader, target_model)

In [12]:
N_pub  = len(pub_ds.ids)
N_priv = len(priv_ds.ids)

In [13]:
SHADOW_DIR = Path("/content/shadow_models")
SHADOW_DIR.mkdir(parents=True, exist_ok=True)

shadow_conf_pub    = []
shadow_conf_priv   = []
shadow_train_masks = []

for s in range(NUM_SHADOW):
    t0 = time.time()

    perm      = np.random.permutation(N_pub)
    n_train   = int(N_pub * SUBSET_RATIO)
    train_idx = perm[:n_train]
    mask      = np.zeros(N_pub, dtype=bool)
    mask[train_idx] = True
    shadow_train_masks.append(mask)

    shadow    = make_model().to(DEVICE)
    optimizer = optim.SGD(shadow.parameters(), lr=SHADOW_LR, momentum=0.9, weight_decay=SHADOW_WD)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SHADOW_EPOCHS)

    pub_ds.transform = transform_train
    train_loader = DataLoader(Subset(pub_ds, train_idx), batch_size=SHADOW_BATCH, shuffle=True, drop_last=True, num_workers=0, pin_memory=True)

    shadow.train()
    for epoch in range(SHADOW_EPOCHS):
        for _, imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            F.cross_entropy(shadow(imgs), labels, label_smoothing=SHADOW_LS).backward()
            optimizer.step()
        scheduler.step()

    shadow.eval()
    pub_confs  = get_confidences(shadow, pub_ds)
    priv_confs = get_confidences(shadow, priv_ds)
    shadow_conf_pub.append(pub_confs)
    shadow_conf_priv.append(priv_confs)

    torch.save(shadow.state_dict(), SHADOW_DIR / f"shadow_{s}.pt")

    elapsed = time.time() - t0
    print(f"  Shadow {s+1}/{NUM_SHADOW} done ({elapsed:.0f}s) "
          f"pub_mean_conf={pub_confs.mean():.4f}")

    del shadow, optimizer, scheduler
    torch.cuda.empty_cache()
    gc.collect()

np.save(SHADOW_DIR / "shadow_conf_pub.npy",    np.stack(shadow_conf_pub))
np.save(SHADOW_DIR / "shadow_conf_priv.npy",   np.stack(shadow_conf_priv))
np.save(SHADOW_DIR / "shadow_train_masks.npy", np.stack(shadow_train_masks))
print("Shadow confidences saved.")


  Shadow 1/1 done (16s) pub_mean_conf=0.1630
Shadow confidences saved.


In [14]:
shadow_pub_stack   = np.stack(shadow_conf_pub,    axis=0)
shadow_priv_stack  = np.stack(shadow_conf_priv,   axis=0)
shadow_train_stack = np.stack(shadow_train_masks, axis=0)

out_mask    = ~shadow_train_stack           # [S, N_pub]
out_counts  = out_mask.sum(axis=0)          # [N_pub]

avg_shadow_pub  = np.zeros(N_pub, dtype=np.float64)
has_out = out_counts > 0
avg_shadow_pub[has_out]  = (shadow_pub_stack[:, has_out] * out_mask[:, has_out]).sum(0) / out_counts[has_out]
avg_shadow_pub[~has_out] = shadow_pub_stack[:, ~has_out].mean(0)
avg_shadow_priv = shadow_priv_stack.mean(axis=0)

log_score_pub  = np.log(target_conf_pub  + 1e-10) - np.log(avg_shadow_pub  + 1e-10)
log_score_priv = np.log(target_conf_priv + 1e-10) - np.log(avg_shadow_priv + 1e-10)


In [15]:
def rmia_score_exact(target_log_scores, ref_log_scores, target_labels, ref_labels, gamma=1.0, same_class=True):
    scores    = np.zeros(len(target_log_scores), dtype=np.float64)
    log_gamma = np.log(gamma + 1e-10)

    if same_class:
        sorted_refs = {c: np.sort(ref_log_scores[ref_labels == c])for c in np.unique(ref_labels)}
        for c in np.unique(target_labels):
            mask = target_labels == c
            refs = sorted_refs.get(c)
            if refs is None or len(refs) == 0:
                continue
            thresholds  = target_log_scores[mask] - log_gamma
            scores[mask] = np.searchsorted(refs, thresholds, side="left") / len(refs)
    else:
        refs       = np.sort(ref_log_scores)
        thresholds = target_log_scores - log_gamma
        scores     = np.searchsorted(refs, thresholds, side="left") / len(refs)

    return scores

rmia_pub = rmia_score_exact(log_score_pub,  log_score_pub,  pub_labels,  pub_labels,  1.5, False)
rmia_priv = rmia_score_exact(log_score_priv, log_score_pub,  priv_labels, pub_labels,  1.5, False)

In [16]:
def normalize_per_class(scores, labels):
    out = np.zeros_like(scores, dtype=np.float64)
    for c in np.unique(labels):
        mask = labels == c
        vals = scores[mask]
        s = vals.std()
        out[mask] = (vals - vals.mean()) / s if s > 0 else 0.0
    return out

ps, qs = rmia_pub, rmia_priv # Changed to use full arrays
method   = (normalize_per_class(ps, pub_labels), normalize_per_class(qs, priv_labels))

In [17]:
def scale_to_01(scores):
    mn, mx = scores.min(), scores.max()
    return (scores - mn) / (mx - mn) if mx > mn else np.full_like(scores, 0.5)

In [18]:
final_scores_priv = scale_to_01(method[1])

In [19]:
priv_ids = priv_ds.ids if isinstance(priv_ds.ids, list) else priv_ds.ids.tolist()

df_submission = pd.DataFrame({"id": priv_ids, "score": final_scores_priv})
df_submission.to_csv(OUTPUT_CSV, index=False)

In [20]:
submission_path = BASE / "submission.csv"

In [21]:
# submit
def die(msg):
    print(msg, file=sys.stderr)
    sys.exit(1)

parser = argparse.ArgumentParser(description="Submit a CSV file to the server.")
args = parser.parse_args([])

submit_path = submission_path

if not submit_path.exists():
    die(f"File not found: {submit_path}")

try:
    with open(submit_path, "rb") as f:
        resp = requests.post(
            f"{BASE_URL}/submit/{TASK_ID}",
            headers={"X-API-Key": API_KEY},
            files={"file": (submit_path.name, f, "application/csv")},
            timeout=(10, 600),
        )
    try:
        body = resp.json()
    except Exception:
        body = {"raw_text": resp.text}

    if resp.status_code == 413:
        die("Upload rejected: file too large (HTTP 413).")

    resp.raise_for_status()

    print("Successfully submitted.")
    print("Server response:", body)
    submission_id = body.get("submission_id")
    if submission_id:
        print(f"Submission ID: {submission_id}")

except requests.exceptions.RequestException as e:
    detail = getattr(e, "response", None)
    print(f"Submission error: {e}")
    if detail is not None:
        try:
            print("Server response:", detail.json())
        except Exception:
            print("Server response (text):", detail.text)
    sys.exit(1)

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Submission error: 401 Client Error: Unauthorized for url: http://34.63.153.158/submit/01-mia
Server response: {'detail': 'Invalid API key'}
Traceback (most recent call last):
  File "/tmp/ipykernel_7208/957261715.py", line 30, in <cell line: 0>
    resp.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 401 Client Error: Unauthorized for url: http://34.63.153.158/submit/01-mia

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_7208/957261715.py", line 46, in <cell line: 0>
    sys.exit(1)
SystemExit: 1

During handling of the above exception, another exception occurred:

Traceback (most recent call last):


TypeError: object of type 'NoneType' has no len()